In [241]:
import pandas as pd
import numpy as np
from google.colab import files
from google.colab import drive
import os

# Hubungkan Google Drive
drive.mount('/content/drive')

print(os.listdir('/content/drive/MyDrive/UAS_ETL'))
# Folder output ETL
folder_path = "/content/drive/MyDrive/UAS_ETL"
os.makedirs(folder_path, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['hasil_etl_dbd_airminum.csv', 'Data_DBD_2023.csv', 'Persentase Rumah Tangga yang Memiliki Akses terhadap Sumber Air Minum Layak Menurut Provinsi dan Klasifikasi Desa, 2023.csv', 'Data_DBD_2021.csv', 'Persentase Rumah Tangga yang Memiliki Akses terhadap Sumber Air Minum Layak Menurut Provinsi dan Klasifikasi Desa, 2024.csv', 'Data_DBD_2022.csv', 'Data_DBD_2024.csv', 'Persentase Rumah Tangga yang Memiliki Akses terhadap Sumber Air Minum Layak Menurut Provinsi dan Klasifikasi Desa, 2022.csv', 'Persentase Rumah Tangga yang Memiliki Akses terhadap Sumber Air Minum Layak Menurut Provinsi dan Klasifikasi Desa, 2021.csv']


In [242]:
# ==================================================
# 2. EXTRACT DATA
# Membaca dataset dari Google Drive
# ==================================================

dbd21 = pd.read_csv('/content/drive/MyDrive/UAS_ETL/Data_DBD_2021.csv')
dbd22 = pd.read_csv('/content/drive/MyDrive/UAS_ETL/Data_DBD_2022.csv')
dbd23 = pd.read_csv('/content/drive/MyDrive/UAS_ETL/Data_DBD_2023.csv')
dbd24 = pd.read_csv('/content/drive/MyDrive/UAS_ETL/Data_DBD_2024.csv')

air21 = pd.read_csv('/content/drive/MyDrive/UAS_ETL/Persentase Rumah Tangga yang Memiliki Akses terhadap Sumber Air Minum Layak Menurut Provinsi dan Klasifikasi Desa, 2021.csv')
air22 = pd.read_csv('/content/drive/MyDrive/UAS_ETL/Persentase Rumah Tangga yang Memiliki Akses terhadap Sumber Air Minum Layak Menurut Provinsi dan Klasifikasi Desa, 2022.csv')
air23 = pd.read_csv('/content/drive/MyDrive/UAS_ETL/Persentase Rumah Tangga yang Memiliki Akses terhadap Sumber Air Minum Layak Menurut Provinsi dan Klasifikasi Desa, 2023.csv')
air24 = pd.read_csv('/content/drive/MyDrive/UAS_ETL/Persentase Rumah Tangga yang Memiliki Akses terhadap Sumber Air Minum Layak Menurut Provinsi dan Klasifikasi Desa, 2024.csv')

In [243]:
# ==================================================
# Membaca data
# ==================================================
print("DBD 2021 :", dbd21.shape)
print("DBD 2022 :", dbd22.shape)
print("DBD 2023 :", dbd23.shape)
print("DBD 2024 :", dbd24.shape)

print("AIR 2021 :", air21.shape)
print("AIR 2022 :", air22.shape)
print("AIR 2023 :", air23.shape)
print("AIR 2024 :", air24.shape)

DBD 2021 : (35, 7)
DBD 2022 : (35, 7)
DBD 2023 : (39, 7)
DBD 2024 : (39, 7)
AIR 2021 : (42, 4)
AIR 2022 : (42, 4)
AIR 2023 : (42, 4)
AIR 2024 : (42, 4)


In [244]:
# ==================================================
# 3. ADD YEAR
# ==================================================

dbd21['Tahun'] = 2021
dbd22['Tahun'] = 2022
dbd23['Tahun'] = 2023
dbd24['Tahun'] = 2024

air21['Tahun'] = 2021
air22['Tahun'] = 2022
air23['Tahun'] = 2023
air24['Tahun'] = 2024

In [245]:
# ==================================================
# 4. CONCAT DATA
# Menggabungkan data seluruh tahun
# ==================================================

dbd = pd.concat(
    [dbd21, dbd22, dbd23, dbd24],
    ignore_index=True
)

air = pd.concat(
    [air21, air22, air23, air24],
    ignore_index=True
)

In [246]:
# ==================================================
# 5. DATA CLEANING
# Menghapus missing value dan data duplikat
# ==================================================

def clean_dataframe(df):

    df = df.dropna()

    df = df.drop_duplicates()

    return df

In [247]:
dbd = clean_dataframe(dbd)
air = clean_dataframe(air)

In [248]:
# ==================================================
# 6. STANDARDIZE DATA
# Menyeragamkan format nama provinsi
# ==================================================

dbd['Provinsi'] = (
    dbd['Provinsi']
    .str.upper()
    .str.strip()
)

air['38 Provinsi'] = (
    air['38 Provinsi']
    .str.upper()
    .str.strip()
)

In [249]:
# ==================================================
# 7. SELECT COLUMN
# Memilih kolom yang dibutuhkan
# ==================================================

dbd = dbd[
    ['Provinsi','Tahun','Jumlah Kasus']
]

air = air[
    ['38 Provinsi','Tahun','Unnamed: 3']
]

In [250]:
# ==================================================
# 8. DATA INTEGRATION
# Menggabungkan data DBD dan Air Minum
# ==================================================

# 1. Hapus agregat nasional
air = air[air['38 Provinsi'] != 'INDONESIA']

# 2. Samakan nama provinsi
air['38 Provinsi'] = air['38 Provinsi'].replace({
    'KEP. BANGKA BELITUNG': 'KEPULAUAN BANGKA BELITUNG',
    'KEP. RIAU': 'KEPULAUAN RIAU'
})

# 3. Cek kesesuaian provinsi
provinsi_dbd = set(dbd['Provinsi'].unique())
provinsi_air = set(air['38 Provinsi'].unique())

hilang = provinsi_dbd - provinsi_air

print("Provinsi yang hilang:")
print(hilang)

# 4. Merge data
data_final = pd.merge(
    dbd,
    air,
    left_on=['Provinsi','Tahun'],
    right_on=['38 Provinsi','Tahun'],
    how='inner'
)

print(f"DEBUG (qRg9bA8SwIGI): Columns in data_final after merge: {data_final.columns.tolist()}")

Provinsi yang hilang:
{'INDONESIA'}
DEBUG (qRg9bA8SwIGI): Columns in data_final after merge: ['Provinsi', 'Tahun', 'Jumlah Kasus', '38 Provinsi', 'Unnamed: 3']


In [251]:
# ==================================================
# 9. DATA QUALITY REPORT
# ==================================================

print("Jumlah Data :", len(data_final))

print(
    "Jumlah Provinsi :",
    data_final['Provinsi'].nunique()
)

print(
    "Rentang Tahun :",
    data_final['Tahun'].min(),
    "-",
    data_final['Tahun'].max()
)

print(
    "Missing Value :",
    data_final.isnull().sum().sum()
)

print(
    "Duplicate Data :",
    data_final.duplicated().sum()
)

print(f"DEBUG (dmhf2gySwKAb): Columns in data_final at end of cell: {data_final.columns.tolist()}")

Jumlah Data : 144
Jumlah Provinsi : 38
Rentang Tahun : 2021 - 2024
Missing Value : 0
Duplicate Data : 0
DEBUG (dmhf2gySwKAb): Columns in data_final at end of cell: ['Provinsi', 'Tahun', 'Jumlah Kasus', '38 Provinsi', 'Unnamed: 3']


In [252]:
# ==================================================
# 10. FINAL TRANSFORMATION
# Membersihkan dan mengganti nama kolom
# ==================================================

# The columns '38 Provinsi' and 'Unnamed: 3' were already handled
# by previous cells (k1fMupPx8bOQ and 5SB6Quylzlo9).
# Therefore, the initial drop and rename operations are redundant here.

print(f"DEBUG (2ppCovgNyDEl): Columns BEFORE rename: {data_final.columns.tolist()}")
print(f"DEBUG (2ppCovgNyDEl): 'Unnamed: 3' in columns BEFORE rename: {'Unnamed: 3' in data_final.columns}")

# Rename 'Unnamed: 3' to 'Akses_Air_Minum' before using it
data_final = data_final.rename(columns={'Unnamed: 3': 'Akses_Air_Minum'})

print(f"DEBUG (2ppCovgNyDEl): Columns AFTER rename: {data_final.columns.tolist()}")
print(f"DEBUG (2ppCovgNyDEl): 'Akses_Air_Minum' in columns AFTER rename: {'Akses_Air_Minum' in data_final.columns}")

data_final['Akses_Air_Minum'] = pd.to_numeric(
    data_final['Akses_Air_Minum'],
    errors='coerce'
)

print(f"DEBUG (2ppCovgNyDEl): Columns AFTER to_numeric: {data_final.columns.tolist()}")

data_final = data_final.dropna()

print(data_final.info())
print(data_final.head())
print(data_final.shape)

print(data_final.isnull().sum())

print(data_final.info())

print(data_final['Provinsi'].nunique())

print(sorted(data_final['Tahun'].unique()))

print("=== DATA QUALITY REPORT ===")

print("Shape :", data_final.shape)

# Diagnostic print to verify data_final's state before 5SB6Quylzlo9
print("Debugging data_final.columns at end of 2ppCovgNyDEl:", data_final.columns)
print("Debugging data_final.shape at end of 2ppCovgNyDEl:", data_final.shape)
print("Debugging data_final.head() at end of 2ppCovgNyDEl:", data_final.head())

DEBUG (2ppCovgNyDEl): Columns BEFORE rename: ['Provinsi', 'Tahun', 'Jumlah Kasus', '38 Provinsi', 'Unnamed: 3']
DEBUG (2ppCovgNyDEl): 'Unnamed: 3' in columns BEFORE rename: True
DEBUG (2ppCovgNyDEl): Columns AFTER rename: ['Provinsi', 'Tahun', 'Jumlah Kasus', '38 Provinsi', 'Akses_Air_Minum']
DEBUG (2ppCovgNyDEl): 'Akses_Air_Minum' in columns AFTER rename: True
DEBUG (2ppCovgNyDEl): Columns AFTER to_numeric: ['Provinsi', 'Tahun', 'Jumlah Kasus', '38 Provinsi', 'Akses_Air_Minum']
<class 'pandas.core.frame.DataFrame'>
Index: 140 entries, 0 to 143
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Provinsi         140 non-null    object 
 1   Tahun            140 non-null    int64  
 2   Jumlah Kasus     140 non-null    int64  
 3   38 Provinsi      140 non-null    object 
 4   Akses_Air_Minum  140 non-null    float64
dtypes: float64(1), int64(2), object(2)
memory usage: 6.6+ KB
None
         Provinsi  Tahun  Ju

In [253]:
# ==================================================
# 11. STANDARDIZE COLUMN NAME
# ==================================================

print("Debugging data_final.columns at start of 5SB6Quylzlo9:", data_final.columns)
print("Debugging data_final.shape at start of 5SB6Quylzlo9:", data_final.shape)

# Direct rename of 'Jumlah Kasus' to 'Jumlah_Kasus'. Will raise KeyError if not found.
data_final = data_final.rename(columns={'Jumlah Kasus': 'Jumlah_Kasus'})

# Direct drop of '38 Provinsi'. Will raise KeyError if not found.
data_final = data_final.drop(columns=['38 Provinsi'])

# Explicitly select and order the final columns required for the next stage.
# This ensures only desired columns are kept and in a specific order.
final_columns_after_5SB6Quylzlo9 = ['Provinsi', 'Tahun', 'Jumlah_Kasus', 'Akses_Air_Minum']

# Filter the DataFrame to include only these columns.
data_final = data_final[final_columns_after_5SB6Quylzlo9]

print("Columns after 5SB6Quylzlo9:", data_final.columns)

Debugging data_final.columns at start of 5SB6Quylzlo9: Index(['Provinsi', 'Tahun', 'Jumlah Kasus', '38 Provinsi', 'Akses_Air_Minum'], dtype='object')
Debugging data_final.shape at start of 5SB6Quylzlo9: (140, 5)
Columns after 5SB6Quylzlo9: Index(['Provinsi', 'Tahun', 'Jumlah_Kasus', 'Akses_Air_Minum'], dtype='object')


In [254]:
# ==================================================
# 12. EXPORT DATA
# Menyimpan hasil ETL ke CSV
# ==================================================

output_file = (
    '/content/drive/MyDrive/UAS_ETL/'
    'hasil_etl_dbd_airminum.csv'
)

data_final.to_csv(
    output_file,
    index=False
)
print("File berhasil disimpan ke Google Drive")


File berhasil disimpan ke Google Drive


In [255]:
# ==================================================
# 13. LOAD DATA TO DATABASE
# Memuat hasil ETL ke MySQL Aiven
# ==================================================
!pip install sqlalchemy psycopg2-binary pymysql

In [256]:
from sqlalchemy import create_engine

def connect_aiven():

    DATABASE_URL = (
        "mysql+pymysql://avnadmin:AVNS_c6K9EHiaWNQ2cyyu6we"
        "@mysql-4c516e3-data-engineering.d.aivencloud.com:19977/defaultdb"
    )

    engine = create_engine(
        DATABASE_URL,
        connect_args={
            "ssl": {
                "ssl_mode": "REQUIRED"
            }
        }
    )

    return engine

In [257]:
from sqlalchemy import text

def create_table(engine):

    with engine.begin() as conn:
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS dbd_airminum (
                id INT PRIMARY KEY,
                Provinsi VARCHAR(100),
                Tahun INT,
                Jumlah_Kasus INT,
                Akses_Air_Minum FLOAT
            )
        """))

    print("Tabel berhasil dibuat")

In [258]:
# This cell is now empty as 'id' column insertion is moved to H8xX5v6orzu7

In [259]:
# ==================================================
# 12. FINAL COLUMN RENAMING AND ORDERING FOR DATABASE
# ==================================================

# Step 1: Rename columns to lowercase for the database
column_rename_map = {
    'Provinsi': 'provinsi',
    'Tahun': 'tahun',
    'Jumlah_Kasus': 'jumlah_kasus',
    'Akses_Air_Minum': 'akses_air_minum'
}
data_final = data_final.rename(columns=column_rename_map)

# Step 2: Explicitly select and order the columns, excluding 'id' which is added later
final_columns_for_db_pre_id = ['provinsi', 'tahun', 'jumlah_kasus', 'akses_air_minum']

# Filter to keep only the final_columns_for_db_pre_id, preserving order
data_final = data_final[final_columns_for_db_pre_id]

print("Columns after F3IwRkvDrGTf:", data_final.columns)
print("Data types after F3IwRkvDrGTf:")
print(data_final.dtypes)

Columns after F3IwRkvDrGTf: Index(['provinsi', 'tahun', 'jumlah_kasus', 'akses_air_minum'], dtype='object')
Data types after F3IwRkvDrGTf:
provinsi            object
tahun                int64
jumlah_kasus         int64
akses_air_minum    float64
dtype: object


In [260]:
from sqlalchemy import text

engine = connect_aiven()

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS dbd_airminum"))

In [261]:
def create_table(engine):

    with engine.begin() as conn:
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS dbd_airminum (
                id INT PRIMARY KEY,
                provinsi VARCHAR(100),
                tahun INT,
                jumlah_kasus INT,
                akses_air_minum FLOAT
            )
        """))

    print("Tabel berhasil dibuat")

In [262]:
def load_data(engine, df):

    df.to_sql(
        "dbd_airminum",
        engine,
        if_exists="append",
        index=False,
        chunksize=1000,
        method="multi"
    )

    print("Data berhasil dikirim ke Aiven")

In [263]:
import pandas as pd

def check_data(engine):

    query = """
    SELECT *
    FROM dbd_airminum
    LIMIT 10
    """

    return pd.read_sql(query, engine)

In [264]:
print(data_final.columns)
print(data_final.dtypes)
print(data_final.columns.tolist())

Index(['provinsi', 'tahun', 'jumlah_kasus', 'akses_air_minum'], dtype='object')
provinsi            object
tahun                int64
jumlah_kasus         int64
akses_air_minum    float64
dtype: object
['provinsi', 'tahun', 'jumlah_kasus', 'akses_air_minum']


In [265]:
# ==================================================
# 14. DATA VERIFICATION AND LOADING
# Memastikan data berhasil masuk ke Aiven
# ==================================================

engine = connect_aiven()

create_table(engine)

# Add 'id' column just before loading to database to ensure it's the first column
# and consistent with the database schema.
if 'id' not in data_final.columns:
    data_final.insert(
        0,
        'id',
        range(1, len(data_final)+1)
    )

# Ensure the data types are correct before loading
data_final['id'] = data_final['id'].astype(int)
data_final['provinsi'] = data_final['provinsi'].astype(str)
data_final['tahun'] = data_final['tahun'].astype(int)
data_final['jumlah_kasus'] = data_final['jumlah_kasus'].astype(int)
data_final['akses_air_minum'] = data_final['akses_air_minum'].astype(float)

load_data(engine, data_final)

print(check_data(engine))

Tabel berhasil dibuat
Data berhasil dikirim ke Aiven
   id                   provinsi  tahun  jumlah_kasus  akses_air_minum
0   1                       ACEH   2021           366            88.79
1   2             SUMATERA UTARA   2021          2918            90.89
2   3             SUMATERA BARAT   2021           654            83.40
3   4                       RIAU   2021          1038            89.76
4   5                      JAMBI   2021           357            79.70
5   6           SUMATERA SELATAN   2021          1135            84.70
6   7                   BENGKULU   2021           628            67.39
7   8                    LAMPUNG   2021          2271            80.20
8   9  KEPULAUAN BANGKA BELITUNG   2021           864            73.40
9  10             KEPULAUAN RIAU   2021          1925            90.83
